# Prepare gene feature for matching analysis
For overlap with genes, we download the Gencode gene set release 50 (https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_50/gencode.v50.basic.annotation.gtf.gz)

In [1]:
import pandas as pd

In [2]:
# columns derived from https://www.gencodegenes.org/pages/data_format.html
gff_df = pd.read_csv(
    "supporting_data/gencode.v50.basic.annotation.gtf.gz",
    compression='gzip',
    delimiter='\t',
    header=4,
    names=[
        'chromosome', 'annotation_source', 'feature_type',
        'genomic_start', 'genomic_end',
        'score', 'genomic_strand', 'genomic_phase', 'additional_info'
    ]
)

In [3]:
gff_df = (
    gff_df
    .query("feature_type == 'gene'")
    .reset_index()
)

In [4]:
# extract additional_info fields; stored as a single string but contains key/value pairs

gff_df['info'] = (
    gff_df['additional_info']
    .str.split('; ') # split main string into key/value attribute strings
    .apply(lambda items: [item.split(' ') for item in items]) # split into key/value tuples
    .apply(dict)
)
info_df = pd.json_normalize(gff_df['info'])
for col in info_df.columns:
    info_df[col] = info_df[col].str.strip('\"') # remove quotation mark characters wrapped around values

In [5]:
gene_df = pd.concat(
    [
        gff_df,
        info_df
    ],
    axis=1
)

In [6]:
gene_df = gene_df.query("gene_type == 'protein_coding'")

In [7]:
gene_df['genomic_start'] = gene_df['genomic_start'].astype(int)
gene_df['genomic_end'] = gene_df['genomic_end'].astype(int)

In [8]:
gene_df = gene_df[['chromosome', 'genomic_start', 'genomic_end', 'gene_id', 'gene_name']]

In [9]:
gene_df['chromosome'] = gene_df['chromosome'].str.strip('chr')

In [10]:
gene_df.to_csv(
    'supporting_data/gene-feature-locations-preprocessed.csv',
    index=False
)